In [ ]:
#import libraries

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [ ]:
#load data
df = pd.read_csv('path/to_your/data.csv', sep=';')
display(df.head())
df.info()

In [ ]:
#Scale n Transform the data
df_numerical = df[['Recency', 'Frequency', 'Monetary']]
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_numerical)

### Determining Optimal Clusters with Elbow Method

The elbow method is used to determine the number of clusters to be made. It works by calculating the Within Cluster Sum of Square (WCSS) or inertia, which indicates how widely spread the data within a cluster.

Usually the graph becomes slopier with the more cluster added, which tells that further cluster addition makes smaller reduction to the inertia. So look for an elbow within the graph.

In [ ]:
#number of clusters by Elbow method
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init = 10)
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

#Plot the graph
plt.plot(range(1, 11), wcss, marker = 'o', linestyle = '--')
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

### Determining Optimal Clusters with Silhouette Score

The Silhouette Score is another valuable metric to determine the optimal number of clusters. It measures how similar an object is to its own cluster (cohesion) compared to other clusters (separation).

The silhouette score ranges from -1 to 1. Score closer to 1 means better data match with their assigned cluster and further with the neighboring clusters.

In [ ]:
# Calculate Silhouette Scores for different numbers of clusters
silhouette_scores = []

for i in range(2, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    score = silhouette_score(df_scaled, kmeans.labels_)
    silhouette_scores.append(score)

# Plot the Silhouette Scores
plt.figure(figsize=(10, 6))
plt.plot(range(2, 11), silhouette_scores, marker='o', linestyle='--')
plt.title('Silhouette Score for Different Numbers of Clusters')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

In [ ]:
#Generate cluster
num_clusters = int(input("Enter the desired number of clusters: "))

kmeans = KMeans(n_clusters=num_clusters, init='k-means++', random_state=42, n_init = 10)
cluster_label = kmeans.fit_predict(df_scaled)
df['cluster'] = cluster_label
print(f"Dataframe with {num_clusters} clusters:\n{df.head()}")

print(f"{num_clusters} clusters distribution:")
display(df['cluster'].value_counts().sort_index())

summary = df.groupby('cluster')[['Recency','Frequency','Monetary']].mean()
summary.style.format({
    'Recency': '{:.1f}',
    'Frequency': '{:.1f}',
    'Monetary': 'Rp {:,.0f}'
})

In [ ]:
#Show pairplot
sns.pairplot(df, vars = ['Recency', 'Frequency', 'Monetary'], hue='cluster')
plt.show()

In [ ]:
# Save the result to an Excel file
%pip install xlsxwriter

output_filename = f'RFM_K_Means{num_clusters}.xlsx'

with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name='Clustered Data', index=False)
    summary.to_excel(writer, sheet_name='Cluster Summary')

print(f"Successfully saved clustered data and summary to {output_filename}")